# Data Cleaning & Preparation

This notebook loads the raw enriched dataset and prepares it for ML training.

## Plan:
1. **Load raw data** — inspect all columns, types, and missing values
2. **Identify columns to drop** — merge keys, duplicates, address fields that served their purpose
3. **Filter rows** — remove non-residential properties, fake transactions, data errors
4. **Clean categories** — fix duplicates, group rare values, standardise casing
5. **Handle missing values** — decide per column: drop rows, impute, or remove feature
6. **Feature engineering** — extract date features, create any new features
7. **Final check** — verify everything is clean, save to parquet for training

In [ ]:
# =============================================================================
# SECTION 1: LOAD RAW DATA
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df = pd.read_parquet("Outputs/lr_epc_towns.parquet")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} — {df[col].dtype} — {df[col].isna().sum():,} NaN ({df[col].isna().mean()*100:.1f}%)")

## Section 2: Column Analysis - What to Keep, Drop, or Clean

There are 53 columns. Each one is categorised as **KEEP**, **DROP**, or **CLEAN** (keep but needs fixing).

### Land Registry (columns 1-16)

| # | Column | Decision | Reason |
|---|--------|----------|--------|
| 1 | transaction_id | DROP | Unique ID, no predictive value |
| 2 | price | **KEEP** | Target variable |
| 3 | date | **KEEP** | Will extract year and month, then drop |
| 4 | postcode_x | DROP | Merge key, location captured by coordinates |
| 5 | property_type_x | DROP - same as property_type_y | D/S/T/F/O — will remove O (commercial) |
| 6 | new_build | **KEEP** | Y/N binary feature |
| 7 | duration | **KEEP** | Freehold/Leasehold |
| 8 | paon | DROP | House number, not a feature |
| 9 | saon | DROP | Flat number, 87% NaN |
| 10 | street | DROP | Address component |
| 11 | locality | DROP | 62% NaN, location captured by coordinates |
| 12 | town | DROP | Location captured by distance features |
| 13 | district | DROP | Location captured by distance features |
| 14 | county | DROP | Location captured by distance features |
| 15 | ppd_category | REMOVE ROW IF = B AND DROP | Land Registry admin code (A=standard, B=additional) |
| 16 | record_status | DROP | Land Registry internal status |

### EPC Matching (columns 17-20)

| # | Column | Decision | Reason |
|---|--------|----------|--------|
| 17 | postcode_clean | DROP | Merge key (postcode without spaces) |
| 18 | address_key | DROP | Merge key (normalised address for matching) |
| 19 | address1 | DROP | Merge key, 26% NaN (no EPC match) |
| 20 | postcode_y | DROP | EPC postcode, same as postcode_x |

### EPC Features (columns 21-33)

| # | Column | Decision | Reason |
|---|--------|----------|--------|
| 21 | uprn | DROP | ID used to get coordinates, not a feature |
| 22 | total_floor_area | **KEEP** | Strong predictor, 26% NaN = no EPC match |
| 23 | lodgement_date | DROP | When EPC was lodged, not when house sold |
| 24 | current_energy_efficiency | **KEEP** | Numeric efficiency score |
| 25 | current_energy_rating | **KEEP** | A-G categorical rating |
| 26 | number_habitable_rooms | **CLEAN** | 37% NaN, has outliers (97+ rooms) |
| 27 | tenure | **CLEAN** | Casing inconsistency (owner-occupied vs Owner-occupied) |
| 28 | property_type_y | **KEEP** | House/Flat/Bungalow/Maisonette (different from property_type_x) |
| 29 | transaction_type | DROP | 19 categories → reason for the epc cert |
| 30 | construction_age_band | **CLEAN** | 74 categories → 13 (individual years → bands) |
| 31 | built_form | **KEEP** | Detached/Semi/Terrace etc |
| 32 | main_fuel | **CLEAN** | 39 categories → 6 (group similar fuel types) |
| 33 | mains_gas_flag | DROP | 43% NaN, too much missing to be useful |

### Coordinates (columns 34-40)

| # | Column | Decision | Reason |
|---|--------|----------|--------|
| 34 | postcode_merge | DROP | Merge key |
| 35 | PCDS | DROP | Merge key |
| 36 | LAT | DROP | Postcode centroid, less precise than exact |
| 37 | LONG | DROP | Postcode centroid, less precise than exact |
| 38 | exact_lat | **KEEP** | Exact property latitude |
| 39 | exact_lon | **KEEP** | Exact property longitude |
| 40 | has_exact_coords | DROP | Flag column, not a feature |

### Spatial Features (columns 41-53)

| # | Column | Decision | Reason |
|---|--------|----------|--------|
| 41 | dist_primary_km | **KEEP** | Distance to nearest primary school |
| 42 | dist_secondary_km | **KEEP** | Distance to nearest secondary school |
| 43 | dist_rail_km | **KEEP** | Distance to nearest rail station |
| 44 | rail_within_1km | **KEEP** | Rail station density |
| 45 | rail_within_5km | **KEEP** | Rail station density |
| 46 | dist_metro_km | **KEEP** | Distance to nearest metro/underground |
| 47 | metro_within_1km | **KEEP** | Metro station density |
| 48 | dist_bus_km | DROP | Near-zero correlation with price (-0.007) |
| 49 | bus_within_1km | DROP | No predictive signal |
| 50 | dist_airport_km | **KEEP** | Distance to nearest airport |
| 51 | dist_coast_km | **KEEP** | Distance to nearest coastline |
| 52 | dist_town_km | **KEEP** | Distance to nearest major town |
| 53 | nearest_town | DROP | Town name, 112 categories, keeping dist only |

### Summary

|  | Count |
|---|---|
| **DROP** | 27 columns (merge keys, IDs, address fields, redundant, high NaN) |
| **KEEP** | 20 columns (1 target + 19 features) |
| **CLEAN** | 6 columns (keep but need fixing before training) |

In [ ]:
# =============================================================================
# IMPORTANT - We are removing rows with certain columns before we drop them. 
# E.g - we are dropping ppd_category = B (those are non standard transactions) before we drop the column
# =============================================================================

"""
print(f"Record status: {df['ppd_category'].value_counts().to_dict()}")
print(df.groupby("ppd_category")["price"].describe())
print(df.groupby("ppd_category")["property_type_x"].value_counts())
dupes = df[df.duplicated(
    subset=["price", "date", "exact_lat", "exact_lon"],
    keep=False
)]
print(dupes["ppd_category"].value_counts())
"""

#Should I drop ppd_category B? - As of now, I am not dropping it. I will keep it for now.
#before = len(df)
#df = df[df["ppd_category"] == "A"]
#print(f"Dropped PPD category B: {before:,} → {len(df):,} (removed {before - len(df):,})")

In [ ]:
drop_columns = [
    # Land Registry — IDs and address fields
    "transaction_id", "postcode_x", "paon", "saon", "street", 
    "locality", "town", "district", "county", 
    "ppd_category", "record_status",
    
    # EPC matching — merge keys
    "postcode_clean", "address_key", "address1", "postcode_y",
    
    # EPC — ID and date
    "uprn", "lodgement_date", "IMD20IND",
    
    # EPC — too much missing
    "mains_gas_flag", 

    # EPC — dropped when investigating and cleaning. Its not needed. 
    "property_type_y", "transaction_type",
    
    # property_type and postcode were cleared accidentally when joining data.
    "property_type","postcode",

    # Coordinates — merge keys and less precise fallbacks
    "postcode_merge", "PCDS", "LAT", "LONG", "has_exact_coords",
    
    # Spatial — no predictive signal
    "dist_bus_km", "bus_within_1km",
    
    # Town name — keeping distance only
    "nearest_town",
]

print(f"Dropping {len(drop_columns)} columns")
print(f"Keeping {len(df.columns) - len(drop_columns)} columns")

before_cols = len(df.columns)
df = df.drop(columns=drop_columns, errors="ignore")
print(f"Dropped {before_cols - len(df.columns)} columns: {before_cols} → {len(df.columns)}")


In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} — {df[col].dtype} — {df[col].isna().sum():,} NaN ({df[col].isna().mean()*100:.1f}%)")

## Section 3: Row Filtering

Before removing any rows, each filter is analysed to understand what we're losing and why. 

Filters to investigate:
1. **No EPC match** - rows without house details (floor area, rooms, energy rating)
2. **No coordinates** -rows without exact property location
3. **Commercial properties** - type O (offices, retail, not residential)
4. **Low prices + High prices** - £1 transfers and non-market transactions - extreme outliers (£5M+)
5. **Floor area errors** - zero, impossibly small, or impossibly larg
6. **Duplicate Transactions** - make sure there is no repeated rows
7. **Energy Efficiency Outliers** - rows with extreme or incorrect values. 

### No EPC match

In [ ]:

no_epc = df[df["total_floor_area"].isna()]
has_epc = df[df["total_floor_area"].notna()]

print(f"No EPC match: {len(no_epc):,} ({len(no_epc)/len(df)*100:.1f}%)")
print(f"Has EPC match: {len(has_epc):,} ({len(has_epc)/len(df)*100:.1f}%)")

print(f"\nWhat we lose without EPC:")
print(f"  These rows have NO: floor area, rooms, energy rating, tenure, built form")
print(f"  They only have: price, property type, date, and spatial features")

print(f"\nPrice comparison:")
print(f"  With EPC    — median: £{has_epc['price'].median():,.0f}, mean: £{has_epc['price'].mean():,.0f}")
print(f"  Without EPC — median: £{no_epc['price'].median():,.0f}, mean: £{no_epc['price'].mean():,.0f}")

print(f"\nDate range without EPC:")
print(f"  {no_epc['date'].min()} to {no_epc['date'].max()}")

# Are they concentrated in certain years?
print(f"\nNo-EPC sales by year:")
print(no_epc["date"].dt.year.value_counts().sort_index())

# Are certain property types more likely to have no EPC?
print(f"\nProperty type — no EPC vs has EPC:")
print(f"  No EPC:  {no_epc['property_type_x'].value_counts().to_dict()}")
print(f"  Has EPC: {has_epc['property_type_x'].value_counts().to_dict()}")

# Geographic — are they spread evenly or concentrated somewhere?
print(f"\nHave coordinates? (can still check location)")
print(f"  No EPC with coords: {no_epc['dist_town_km'].notna().sum():,}")
print(f"  No EPC without coords: {no_epc['dist_town_km'].isna().sum():,}")

# New builds — should always have EPC since they need one by law
print(f"\nNew builds without EPC: {no_epc[no_epc['new_build'] == 'Y'].shape[0]:,}")
print(f"  (New builds require EPC by law)")


In [ ]:
before = len(df)
df = df[df["total_floor_area"].notna()]

### No coordinates

In [ ]:
no_coords = df[df["exact_lat"].isna()]
has_coords = df[df["exact_lat"].notna()]

print(f"No exact coordinates: {len(no_coords):,} ({len(no_coords)/len(df)*100:.1f}%)")
print(f"Has exact coordinates: {len(has_coords):,} ({len(has_coords)/len(df)*100:.1f}%)")


In [ ]:
before = len(df)
df = df[df["exact_lat"].notna()]
print(f"Dropped no coordinates: {before:,} → {len(df):,} (removed {before - len(df):,})")

### Commercial properties

In [ ]:
before = len(df)
df = df[df["property_type_x"] != "O"]
print(f"Dropped type O: {before:,} → {len(df):,} (removed {before - len(df):,})")

### Low prices + High prices

In [ ]:
# look at the distribution of prices under £50k
under_100k = df[df["price"] < 100_000]["price"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(under_100k, bins=200, color="#2E86C1", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Price (£)")
ax.set_ylabel("Count")
ax.set_title(f"Price Distribution Under £100k ({len(under_100k):,} sales)")
plt.tight_layout()
plt.show()

# Print counts in small bands
print(f"Price bands:")
for low, high in [(0, 1000), (1000, 5000), (5000, 10000), (10000, 15000), (15000, 20000), (20000, 30000), (30000, 50000), (50000, 100000)]:
    count = df["price"].between(low, high).sum()
    print(f"  £{low:,}-{high:,}: {count:,}")



In [ ]:
# Look at the distribution of prices over £1M
over_1m = df[df["price"] > 1_000_000]["price"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(over_1m, bins=200, color="#E24B4A", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Price (£)")
ax.set_ylabel("Count")
ax.set_title(f"Price Distribution Over £1M ({len(over_1m):,} sales)")
plt.tight_layout()
plt.show()

# Print counts in bands
print(f"Price bands:")
for low, high in [(1_000_000, 1_500_000), (1_500_000, 2_000_000), 
                   (2_000_000, 3_000_000), (3_000_000, 5_000_000), 
                   (5_000_000, 7_500_000), (7_500_000, 10_000_000), 
                   (10_000_000, 20_000_000), (20_000_000, 50_000_000),
                   (50_000_000, 100_000_000)]:
    count = df["price"].between(low, high).sum()
    print(f"  £{low/1_000_000:.1f}M-{high/1_000_000:.1f}M: {count:,}")

In [ ]:
before = len(df)
df = df[df["price"] >= 10_000]
print(f"Dropped price < £10k: {before:,} → {len(df):,} (removed {before - len(df):,})")

before = len(df)
df = df[df["price"] <= 5_000_000]
print(f"Dropped price > £5M: {before:,} → {len(df):,} (removed {before - len(df):,})")

### Floor area errors

In [ ]:
print(f"Floor area distribution:")
print(df["total_floor_area"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(df["total_floor_area"], bins=200, color="#2E86C1", edgecolor="white", linewidth=0.3)
axes[0].set_title("Floor Area Distribution (all)")
axes[0].set_xlabel("Floor Area (m²)")
axes[0].set_ylabel("Count")
axes[0].axvline(df["total_floor_area"].median(), color="black", linestyle="--", 
                label=f"Median: {df['total_floor_area'].median():.0f}m²")
axes[0].legend()

# Zoom into extremes
extremes = df[(df["total_floor_area"] < 15) | (df["total_floor_area"] > 400)]
axes[1].hist(extremes["total_floor_area"], bins=100, color="#E24B4A", edgecolor="white", linewidth=0.3)
axes[1].set_title(f"Extreme Floor Areas ({len(extremes):,} properties)")
axes[1].set_xlabel("Floor Area (m²)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

# Bands
print(f"\nFloor area bands:")
for low, high in [(0, 5), (5, 10), (10, 15), (15, 30), (30, 50), 
                   (50, 100), (100, 200), (200, 500), (500, 1000), (1000, 10000)]:
    count = df["total_floor_area"].between(low, high).sum()
    print(f"  {low}-{high}m²: {count:,}")

# What are the tiny ones?
tiny = df[df["total_floor_area"] < 10]
print(f"\nUnder 10m² ({len(tiny):,} properties):")
print(tiny[["price", "property_type_x", "total_floor_area", "built_form"]].to_string())

# What are the huge ones?
huge = df[df["total_floor_area"] > 500]
print(f"\nOver 500m² ({len(huge):,} properties):")
print(huge[["price", "property_type_x", "total_floor_area", "built_form"]].head(20).to_string())

In [ ]:
before = len(df)
df = df[(df["total_floor_area"] >= 10) & (df["total_floor_area"] <= 1000)]
print(f"Dropped floor area < 10 or > 1000m²: {before:,} → {len(df):,} (removed {before - len(df):,})")

### Duplicate Transactions

In [ ]:
# Checking how many transactions are fully duplicated (all columns are the same)
dupes = df.duplicated().sum()
print(f"\n {dupes} fully-duplicated rows.")

group_sizes = df.groupby(list(df.columns),dropna=False).size()
repeat_dist = group_sizes[group_sizes > 1].value_counts().sort_index()
print(repeat_dist)

dupe_rows = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print(f"{len(dupe_rows):,} rows involved in duplication")
print(dupe_rows.head(10).to_string())

In [ ]:
# DROPPING FULLY DUPLICATED ROWS
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"{before:,} -> {len(df):,} (dropped {before - len(df):,})")

In [ ]:
# Same property, same price, same date = likely duplicate record
dupes = df.duplicated(subset=["price", "date", "exact_lat", "exact_lon"], keep=False)
print(f"Potential duplicates: {dupes.sum():,} ({dupes.sum()/len(df)*100:.2f}%)")

# Look at them
if dupes.sum() > 0:
    dupe_df = df[dupes].sort_values(["exact_lat", "exact_lon", "date"])
    print(f"\nExamples:")
    print(dupe_df[["price", "date", "property_type_x", "total_floor_area", 
                    "built_form", "exact_lat", "exact_lon"]].head(20).to_string())
    
    # How many unique properties are involved?
    unique_locations = dupe_df.groupby(["exact_lat", "exact_lon"]).size()
    print(f"\nDuplicate groups: {len(unique_locations):,}")
    print(f"Entries per group:")
    print(unique_locations.value_counts().sort_index())
    
    # Are they exact duplicates (all columns same) or just same price/date/location?
    exact_dupes = df.duplicated(keep=False)
    print(f"\nExact row duplicates (all columns): {exact_dupes.sum():,}")

In [ ]:
# What property has 114 duplicate entries? ----> ITS A FLAT! 
dupe_df = df[dupes].sort_values(["exact_lat", "exact_lon", "date"])
groups = dupe_df.groupby(["exact_lat", "exact_lon", "price", "date", "property_type_x"]).size()
biggest = groups.sort_values(ascending=False).head(5)
print("Biggest duplicate groups:")
print(biggest)

In [ ]:
raw = pd.read_parquet("Outputs/lr_epc_towns.parquet")
dev = raw[
    (raw["exact_lat"].between(52.574, 52.575)) & 
    (raw["exact_lon"].between(-0.238, -0.237))
]
print(dev[["paon", "saon", "street", "postcode_x", "price", "property_type_x", "total_floor_area"]].head(5).to_string())

In [ ]:
# Non-flat properties sharing exact coordinates
non_flat = df[df["property_type_x"] != "F"]
location_counts = non_flat.groupby(["exact_lat", "exact_lon", "date"]).size()
shared = location_counts[location_counts > 1]

print(f"Non-flat locations with multiple properties: {len(shared):,}")
print(f"Total properties at shared locations: {shared.sum():,}")
print(f"\nHow many share each location:")
print(shared.value_counts().sort_index())

In [ ]:
# Get shared locations with same date
non_flat = df[df["property_type_x"] != "F"]
location_date_counts = non_flat.groupby(["exact_lat", "exact_lon", "date"]).size()
shared = location_date_counts[location_date_counts > 1]

print(f"Same location + same date (non-flat): {len(shared):,} groups")
print(f"Total properties: {shared.sum():,}")

# Pull samples from raw data
raw = pd.read_parquet("Outputs/lr_epc_towns.parquet")

samples = shared.sample(min(100, len(shared)), random_state=42)
for (lat, lon, date), count in samples.items():
    matches = raw[
        (raw["exact_lat"].between(lat - 0.0001, lat + 0.0001)) & 
        (raw["exact_lon"].between(lon - 0.0001, lon + 0.0001)) &
        (raw["date"] == date) &
        (raw["property_type_x"] != "F")
    ]
    print(f"\n--- {count} properties at ({lat:.6f}, {lon:.6f}) on {date.date()} ---")
    print(matches[["paon", "street", "postcode_x", "price", "date",
                    "property_type_x", "total_floor_area", "uprn", "new_build"]].head(10).to_string())

In [ ]:
# Merge approach — fast, no row-by-row loops. Should take 2-3 minutes.
""" commenting out for speed - this part is only analytical anyway
# Step 1: Find duplicate lat/lon/date in cleaned data (non-flat)
non_flat = df[df["property_type_x"] != "F"].copy()
non_flat["loc_date_key"] = (
    non_flat["exact_lat"].round(6).astype(str) + "_" + 
    non_flat["exact_lon"].round(6).astype(str) + "_" + 
    non_flat["date"].astype(str)
)
counts = non_flat["loc_date_key"].value_counts()
dupe_keys = set(counts[counts > 1].index)
print(f"Duplicate groups: {len(dupe_keys):,}")

# Step 2: Build same key in raw data
raw = pd.read_parquet("Outputs/lr_epc_towns.parquet")
raw = raw[raw["property_type_x"] != "F"]
raw["loc_date_key"] = (
    raw["exact_lat"].round(6).astype(str) + "_" + 
    raw["exact_lon"].round(6).astype(str) + "_" + 
    raw["date"].astype(str)
)

# Step 3: Filter raw to only duplicate keys
dupes_raw = raw[raw["loc_date_key"].isin(dupe_keys)].sort_values("loc_date_key")

# Step 4: Add empty row between groups
groups = []
for key, group in dupes_raw.groupby("loc_date_key"):
    groups.append(group)
    groups.append(pd.DataFrame([{}]))

result = pd.concat(groups, ignore_index=True)
result = result.drop(columns=["loc_date_key"])

result.to_csv("Outputs/location_date_duplicates.csv", index=False)
print(f"Exported {len(result):,} rows, all columns")
print(f"Saved to Outputs/location_date_duplicates.csv")
"""

In [ ]:
# Analyse differences between duplicates in each group
"""
dupes_raw = raw[raw["loc_date_key"].isin(dupe_keys)].sort_values("loc_date_key")

identical_groups = 0
diff_groups = 0
diff_summary = {}  # which columns differ and how often

for key, group in dupes_raw.groupby("loc_date_key"):
    # Find columns that differ within this group
    differing_cols = []
    for col in group.columns:
        if col == "loc_date_key":
            continue
        unique_vals = group[col].dropna().unique()
        if len(unique_vals) > 1:
            differing_cols.append(col)
    
    if len(differing_cols) == 0:
        identical_groups += 1
    else:
        diff_groups += 1
        for col in differing_cols:
            diff_summary[col] = diff_summary.get(col, 0) + 1

print(f"Completely identical (true duplicates): {identical_groups:,}")
print(f"Have differences: {diff_groups:,}")

print(f"\nWhich columns differ and how often:")
sorted_diffs = sorted(diff_summary.items(), key=lambda x: -x[1])
for col, count in sorted_diffs:
    print(f"  {col}: differs in {count:,} groups ({count/(identical_groups+diff_groups)*100:.1f}%)")
"""

In [ ]:
# Remove exact duplicate property records while retaining one representative observation.
# Duplicates are identified using sale price, transaction date, property coordinates,
# and duration. 
# 
# Including duration ensures that records differing only in
# freehold/leasehold status are preserved, as they may represent distinct observations
# rather than true duplicates.

# Create the mask
mask = (
    (df["property_type_x"] == "F") |
    ~df.duplicated(
        subset=[
            "price",
            "date",
            "exact_lat",
            "exact_lon",
            "duration"
        ],
        keep="first"
    )
)

# Rows that will be dropped
dropped = df[~mask].copy()


#dropping all duplicates - including FLATS - which are not included in the mask above.
df = df.drop_duplicates().reset_index(drop=True)


print(f"Rows before cleaning: {len(df):,}")
print(f"Rows dropped: {len(dropped):,}")
print(f"Rows after cleaning: {mask.sum():,}")


##### NOTE: 
Im only dropping full duplicates across all columns. I looked into dropping rows that share the same price, sale date, and location, but the majority of those turned out to be flats, which legitimately share a coordinate. There are also houses in estates that can have the same location and sale date without being duplicates. That is my justification for keeping rows that only match on some features rather than all of them.

### Energy Efficiency Outliers

In [ ]:
# Summary statistics
print(df["current_energy_efficiency"].describe())

# Check for missing values
print(f"Missing values: {df['current_energy_efficiency'].isna().sum():,}")


# Histogram
plt.figure(figsize=(8, 4))
plt.hist(df["current_energy_efficiency"].dropna(), bins=50)
plt.title("Distribution of Energy Efficiency Scores")
plt.xlabel("Energy Efficiency Score")
plt.ylabel("Frequency")
plt.show()

## Section 4: Clean categories

This section examines all categorical variables for inconsistent labels, duplicate categories (e.g., differences in spelling or capitalisation), infrequent categories, and non-informative values.

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns

for col in categorical_cols:
    print(f"\n{'='*70}")
    print(f"{col} ({df[col].nunique()} unique values)")
    print(df[col].value_counts(dropna=False).sort_index())

In [ ]:
print(df["construction_age_band"].dtype)
unique_values = sorted(df["construction_age_band"].dropna().unique())

for value in unique_values:
    print(value)

In [ ]:
# =============================================================================
# CLEAN — CONSTRUCTION AGE
# =============================================================================

# We are creating 3 new columns here. construction year band 1990-2000, exact year 1997, and bool if exact year exists.
# bool is true if exact data was provided nad false if it was taken from range  

# Some EPC records store age bands as RdSAP letter codes (A-L) instead of
# date ranges. These are the official SAP assessment codes defined by BRE.
# Source: https://files.bregroup.com/SAP/RdSAP10-dt13.02.2024.pdf (Table 1)

rdsap_codes = {
    "A": "before 1900", "B": "1900-1929", "C": "1930-1949",
    "D": "1950-1966", "E": "1967-1975", "F": "1976-1982",
    "G": "1983-1990", "H": "1991-1995", "I": "1996-2002",
    "J": "2003-2006", "K": "2007-2011", "L": "2012 onwards",
}

# Representative years are ONLY used when an exact construction year is not
# available. Exact years are always preserved where they exist.

construction_year_map = {
    "before 1900": 1890, "1900-1929": 1915, "1930-1949": 1940,
    "1950-1966": 1958, "1967-1975": 1971, "1976-1982": 1979,
    "1983-1990": 1986, "1991-1995": 1993, "1996-2002": 1999,
    "2003-2006": 2004.5, "2007-2011": 2009, "2012 onwards": 2018,
    "Unknown": np.nan,
}

s = df["construction_age_band"].astype("string").str.strip()
s = s.str.replace("England and Wales: ", "", regex=False)
s = s.replace(rdsap_codes)
s = s.replace({
    "2007 onwards": "2012 onwards",
    "2012-2021": "2012 onwards",
    "2022 onwards": "2012 onwards",
})

# Values that are not correct construction years are treated as Unknown. eg 102, 200, 2204
# Valid exact years are 1600-2026.
numeric_values = pd.to_numeric(s, errors="coerce")
invalid_numeric = (
    numeric_values.notna() &
    ~numeric_values.between(1600, 2026)
)
s.loc[invalid_numeric] = "Unknown"

# Preserve exact four-digit years where available.
# Examples: 1796 -> 1796, 1848 -> 1848, 1952 -> 1952, 2004 -> 2004
exact_year = pd.to_numeric(
    s.where(s.str.fullmatch(r"\d{4}", na=False)),
    errors="coerce"
)

# Sanity check: only accept plausible construction years
exact_year = exact_year.where(exact_year.between(1600, 2026))

def year_to_band(year):
    if pd.isna(year): return pd.NA
    year = int(year)
    if year < 1900: return "before 1900"
    elif year < 1930: return "1900-1929"
    elif year < 1950: return "1930-1949"
    elif year < 1967: return "1950-1966"
    elif year < 1976: return "1967-1975"
    elif year < 1983: return "1976-1982"
    elif year < 1991: return "1983-1990"
    elif year < 1996: return "1991-1995"
    elif year < 2003: return "1996-2002"
    elif year < 2007: return "2003-2006"
    elif year < 2012: return "2007-2011"
    else: return "2012 onwards"


# Generate a band from the exact year where one is available
exact_year_band = exact_year.apply(year_to_band)

# Exact years take priority. Otherwise, retain the cleaned original band.
df["construction_age_band"] = exact_year_band.fillna(s)
# Any remaining missing values become "Unknown"
df["construction_age_band"] = df["construction_age_band"].fillna("Unknown")


# Exact construction years take priority.
df["construction_year"] = exact_year.astype("float64")
# If no exact year exists, use the representative year for the age band.
df["construction_year"] = df["construction_year"].fillna(
    df["construction_age_band"].map(construction_year_map)
)
df["construction_year"] = pd.to_numeric(df["construction_year"], errors="coerce")

# True  = actual construction year was supplied by the EPC
# False = year is estimated from an age band
df["construction_year_exact"] = exact_year.notna()


# check results 
print(f"Number of construction age bands: {df['construction_age_band'].nunique()}")

print("\nConstruction age bands:")
print(df["construction_age_band"].value_counts(dropna=False).sort_index())

print("\nExact construction years:")
print(df["construction_year_exact"].value_counts(dropna=False))
print(f"Construction year NaN: {df['construction_year'].isna().sum():,}")
print("\nConstruction year summary:")
print(df["construction_year"].describe())

print(
    df[
        [
            "construction_age_band",
            "construction_year",
            "construction_year_exact"
        ]
    ].head(20)
)

# dropping the unknown construction age band rows. there is only 68 of them and they are not useful for analysis.
df = df[df["construction_age_band"] != "Unknown"].copy() 

print(f"Rows remaining: {len(df):,}")
print(f"Unknown bands remaining: {(df['construction_age_band'] == 'Unknown').sum():,}")

In [ ]:
df = df.dropna(subset=["construction_age_band"]) #dropping rows since there is only two

In [ ]:
# =============================================================================
# SECTION 5.2: CLEAN — MAIN FUEL
# =============================================================================

# 48 categories in raw data, many are the same fuel in different formats
# e.g. "mains gas (not community)" and "Gas: mains gas" are both mains gas.
# Grouped by actual fuel type.

def clean_fuel(val):
    if pd.isna(val):
        return val
    val = val.lower()
    if "mains gas" in val or ("gas" in val and "lpg" not in val and "bio" not in val):
        return "mains_gas"
    elif "electric" in val:
        return "electricity"
    elif "oil" in val and "bio" not in val:
        return "oil"
    elif "lpg" in val or "lng" in val:
        return "lpg"
    elif "coal" in val or "anthracite" in val or "smokeless" in val or "dual fuel" in val:
        return "solid_fuel"
    elif ("wood" in val or "biomass" in val or "biogas" in val 
          or "biodiesel" in val or "bioethanol" in val):
        return "biomass"
    else:
        return "other"

df["main_fuel"] = df["main_fuel"].apply(clean_fuel)

print(f"Categories: {df['main_fuel'].nunique()}")
print(df["main_fuel"].value_counts())

In [ ]:
# =============================================================================
# SECTION 5.3: CLEAN — Tenure + Built Form
# =============================================================================

df["tenure"] = ( df["tenure"].str.strip().str.lower())

print("=" * 60)
print("Tenure (after cleaning)")
print(df["tenure"].value_counts(dropna=False))

df["built_form"] = df["built_form"].replace({
    "Enclosed End-Terrace": "End-Terrace",
    "Enclosed Mid-Terrace": "Mid-Terrace",
    "Not Recorded": "Unknown",
})

print("=" * 60)
print("Built Form (after cleaning)")
print(df["built_form"].value_counts(dropna=False))

In [ ]:
# see everything is cleaned 

categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns

for col in categorical_cols:
    print(f"\n{'='*70}")
    print(f"{col} ({df[col].nunique()} unique values)")
    print(df[col].value_counts(dropna=False).sort_index())

## Section 5: Fill in missing values.

In [ ]:
print("Remaining NaN after filtering and cleaning:\n")
nans = df.isnull().sum()
nans = nans[nans > 0].sort_values(ascending=False)

for col, count in nans.items():
    print(f"  {col}: {count:,} ({count/len(df)*100:.1f}%)")

print(f"\nTotal rows: {len(df):,}")

### number of rooms 

In [ ]:
# =============================================================================
# SECTION 5.1: INVESTIGATE — number_habitable_rooms
# =============================================================================

rooms = df["number_habitable_rooms"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of all room counts
axes[0].hist(rooms, bins=100, color="#2E86C1", edgecolor="white", linewidth=0.3)
axes[0].set_title(f"Room Count Distribution (all)")
axes[0].set_xlabel("Rooms")
axes[0].set_ylabel("Count")

# Zoomed — under 15 rooms (where the real data lives)
axes[1].hist(rooms[rooms <= 15], bins=15, color="#1D9E75", edgecolor="white", linewidth=0.3)
axes[1].set_title(f"Room Count Distribution (≤15)")
axes[1].set_xlabel("Rooms")
axes[1].set_ylabel("Count")

# Rooms vs floor area — does it make sense?
sample = df[df["number_habitable_rooms"].notna()].sample(50000, random_state=42)
axes[2].scatter(sample["total_floor_area"], sample["number_habitable_rooms"], 
                s=0.5, alpha=0.2, color="#2E86C1")
axes[2].set_title("Floor Area vs Rooms")
axes[2].set_xlabel("Floor Area (m²)")
axes[2].set_ylabel("Rooms")

plt.tight_layout()
plt.show()

# Counts by room number
print("Room counts:")
print(rooms.value_counts().sort_index().to_string())

# Suspicious ones — high rooms, small floor area
suspicious = df[(df["number_habitable_rooms"] > 15) & (df["total_floor_area"] < 200)]
print(f"\nSuspicious: >15 rooms but <200m²: {len(suspicious):,}")
print(suspicious[["price", "total_floor_area", "number_habitable_rooms", 
                   "property_type_x", "built_form"]].head(20).to_string())

In [ ]:
# =============================================================================
# SECTION 5.1a: CLEAN — room count outliers
# =============================================================================

# Find the threshold from the data — what's the minimum realistic m² per room?
has_both = df["number_habitable_rooms"].notna() & df["total_floor_area"].notna()
m2_per_room = df.loc[has_both, "total_floor_area"] / df.loc[has_both, "number_habitable_rooms"]

threshold = m2_per_room.quantile(0.001)
print(f"1st percentile of m² per room: {threshold:.1f}")
print(f"99% of properties have at least {threshold:.1f}m² per room on average")

# Any property below this threshold has a suspicious room count
bad_ratio = (
    (df["total_floor_area"] / df["number_habitable_rooms"]) < threshold
) & df["number_habitable_rooms"].notna()

print(f"\nProperties below threshold: {bad_ratio.sum():,}")

# Show examples of what we're fixing
if bad_ratio.sum() > 0:
    print(f"\nExamples being fixed:")
    print(df.loc[bad_ratio, ["price", "total_floor_area", "number_habitable_rooms",
                              "property_type_x"]].sort_values("number_habitable_rooms", ascending=False).head(300).to_string())

df.loc[bad_ratio, "number_habitable_rooms"] = np.nan
print(f"\nMax rooms now: {df['number_habitable_rooms'].max():.0f}")

In [ ]:
# =============================================================================
# SECTION 5.1: HANDLE — number_habitable_rooms (15.5% missing)
# =============================================================================

print("number_habitable_rooms:")
print(f"  Missing: {df['number_habitable_rooms'].isna().sum():,}")
print(f"  Max: {df['number_habitable_rooms'].max()}")

# Fill missing based on floor area bins
has_both = df["number_habitable_rooms"].notna() & df["total_floor_area"].notna()
bins = [0, 30, 50, 70, 90, 120, 160, 200, 500, 1000]
df["area_bin"] = pd.cut(df["total_floor_area"], bins=bins)
median_rooms = df.loc[has_both].groupby("area_bin")["number_habitable_rooms"].median()

print(f"\n  Median rooms per floor area:")
for bin_range, rooms in median_rooms.items():
    print(f"    {bin_range}: {rooms:.0f} rooms")

missing = df["number_habitable_rooms"].isna()
df.loc[missing, "number_habitable_rooms"] = df.loc[missing, "area_bin"].map(median_rooms)
df = df.drop(columns=["area_bin"])

print(f"\n  After: {df['number_habitable_rooms'].isna().sum():,} still missing")
print(f"  Max rooms: {df['number_habitable_rooms'].max():.0f}")

In [ ]:
# check after cleaning and filling missing values

rooms = df["number_habitable_rooms"].dropna()

# Counts by room number
print("Room counts:")
print(rooms.value_counts().sort_index().to_string())

# Suspicious ones — high rooms, small floor area
suspicious = df[(df["number_habitable_rooms"] > 15) & (df["total_floor_area"] < 200)]
print(f"\nSuspicious: >15 rooms but <200m²: {len(suspicious):,}")
print(suspicious[["price", "total_floor_area", "number_habitable_rooms", 
                   "property_type_x", "built_form"]].head(50).to_string())

suspicious = df[(df["number_habitable_rooms"] > 15)]
print(f"\nAnything with >15 rooms: {len(suspicious):,}")
print(suspicious[["price", "total_floor_area", "number_habitable_rooms", 
                   "property_type_x", "built_form"]].to_string())

### tenure

In [ ]:
df["tenure"] = df["tenure"].fillna("unknown")

### built_form

In [ ]:
df["built_form"] = df["built_form"].fillna("Unknown")

### main_fuel

In [ ]:
df["main_fuel"] = df["main_fuel"].fillna("unknown")

## Feature engineering

In [ ]:
#property_age

sale_year = pd.to_datetime(df["date"], errors="coerce").dt.year

# Calculate property age at sale
df["property_age"] = sale_year - df["construction_year"]

print(df["property_age"].describe())
print(f"\nNaN count: {df['property_age'].isna().sum():,}")
print(f"NaN percentage: {df['property_age'].isna().mean() * 100:.2f}%")

In [ ]:
# Number of months since the first transaction
df["sale_time"] = (
    (df["date"].dt.year - df["date"].dt.year.min()) * 12
    + (df["date"].dt.month - 1)
)

print(f"Sale time: {df['sale_time'].min()} — {df['sale_time'].max()} months")

# Drop original date column
df = df.drop(columns=["date"])

In [ ]:
# Removing day of the year caused some duplicates, so we need to check for duplicates again and remove them
print(f"full_dupes={df.duplicated().sum():,} | rows={len(df):,}")

print(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(4).to_string())

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"{before:,} -> {len(df):,} (dropped {before - len(df):,})")

In [ ]:
distance_cols = [
    "dist_primary_km",
    "dist_secondary_km",
    "dist_rail_km",
    "dist_metro_km",
    "dist_airport_km",
    "dist_coast_km",
    "dist_town_km",
]

for col in distance_cols:
    df[col] = np.log1p(df[col])

## Final check and Save

In [ ]:
# =============================================================================
# Final Data Quality Checks
# =============================================================================

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

print("\nData types:")
print(df.dtypes)

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(3)

missing_summary = (
    pd.DataFrame({
        "Missing": missing,
        "Percent": missing_pct
    })
    .query("Missing > 0")
    .sort_values("Missing", ascending=False)
)

print(missing_summary)

print("\n" + "=" * 70)
print("DUPLICATES")
print("=" * 70)

duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicates:,}")

print("\n" + "=" * 70)
print("NUMERIC SUMMARY")
print("=" * 70)

print(df.describe())

print("\n" + "=" * 70)
print("CATEGORICAL SUMMARY")
print("=" * 70)

categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

In [ ]:
df.to_parquet( "Outputs/clean_property_data.parquet", index=False,compression="snappy")

print("Dataset successfully saved.")